In [1]:
import numpy as np
import pandas as pd
import polars as pl
from mars.analysis.evaluator import profile_risk

# =================================================================
# 1. 构造更真实的模拟数据 (20个特征, 适中区分度)
# =================================================================
np.random.seed(2024)
n_samples = 200000

# --- A. 基础信息 ---
# 日期: 申请时间 (近1年)
dates = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')
apply_date = np.random.choice(dates, n_samples)

# 1. age: 年龄 (20-60), 倒U型风险 (极年轻和极老风险稍高)
age = np.random.randint(20, 60, n_samples)

# 2. gender: 性别 (0/1), 弱特征
gender = np.random.choice(['M', 'F'], n_samples)

# 3. education: 学历 (Ordinal), 负相关
edu_map = {'HighSchool': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
education = np.random.choice(list(edu_map.keys()), n_samples, p=[0.3, 0.5, 0.15, 0.05])
edu_score = np.array([edu_map[x] for x in education])

# --- B. 资产/收入类 (强特征, 负相关) ---
# 4. income_annual: 年收入 (对数正态, 长尾)
income_annual = np.random.lognormal(11, 0.8, n_samples)

# 5. house_value: 房产估值 (大量0值代表无房)
has_house = np.random.binomial(1, 0.4, n_samples)
house_value = has_house * np.random.lognormal(14, 0.5, n_samples)

# 6. debt_ratio: 负债率 (0-1+), U型风险 (太低无信用记录, 太高还款压力大)
debt_ratio = np.random.beta(2, 5, n_samples)

# --- C. 历史信用类 (最强特征, 正相关) ---
# 7. max_overdue_days: 历史最大逾期天数
max_overdue_days = np.random.exponential(10, n_samples).astype(int)

# 8. overdue_cnt_1y: 过去1年逾期次数 (泊松分布)
overdue_cnt_1y = np.random.poisson(0.5, n_samples)

# 9. credit_score_ext: 外部信用分 (正态分布, 高好低坏)
credit_score_ext = np.random.normal(600, 50, n_samples)

# 10. active_loans: 当前在贷笔数
active_loans = np.random.poisson(2, n_samples)

# --- D. 行为/多头类 (中等特征, 正相关) ---
# 11. query_cnt_3m: 近3个月查询次数
query_cnt_3m = np.random.negative_binomial(1, 0.2, n_samples)

# 12. query_org_cnt: 查询机构数
query_org_cnt = np.random.poisson(3, n_samples)

# 13. util_rate: 授信额度使用率
util_rate = np.random.uniform(0, 1.2, n_samples)

# 14. phone_in_net_months: 手机在网时长 (越长越稳定)
phone_in_net_months = np.random.randint(1, 120, n_samples)

# --- E. 弱特征/噪音类 (检验筛选能力) ---
# 15. height: 身高 (纯噪音, IV应接近0)
height = np.random.normal(170, 10, n_samples)

# 16. last_login_hour: 最后登录时间 (0-23, 可能半夜登录风险略高)
last_login_hour = np.random.randint(0, 24, n_samples)

# 17. device_type: 设备类型
device_type = np.random.choice(['iOS', 'Android', 'Other'], n_samples, p=[0.4, 0.55, 0.05])

# 18. is_email_verified: 邮箱是否验证
is_email_verified = np.random.choice([0, 1], n_samples, p=[0.2, 0.8])

# 19. app_install_cnt: APP安装数量
app_install_cnt = np.random.normal(50, 20, n_samples)

# 20. city_tier: 城市等级 (1-5)
city_tier = np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.1, 0.2, 0.3, 0.2, 0.2])

# =================================================================
# 2. 构造 Target (控制 IV 和 KS 在合理范围)
# =================================================================
# 线性组合 + 大量噪音
# 正系数代表风险增加，负系数代表风险降低
logit = (
    - 0.05 * (age - 35) / 10 
    - 0.3 * (edu_score) 
    - 0.4 * np.log1p(income_annual) / 2
    + 0.5 * debt_ratio
    + 0.6 * np.log1p(max_overdue_days) 
    + 0.5 * overdue_cnt_1y 
    - 0.6 * (credit_score_ext - 600) / 50 
    + 0.3 * np.log1p(query_cnt_3m)
    - 0.2 * (phone_in_net_months / 12)
    + 0.1 * (last_login_hour > 22) # 半夜登录微弱风险
    + np.random.normal(0, 2.5, n_samples) # <--- 关键：增加噪音方差以降低 IV/KS
)

# 转换为概率
p = 1 / (1 + np.exp(-logit))

# 设定坏账率约为 5%
threshold = np.percentile(p, 95)
y_30d = (p > threshold).astype(int)

# 构造 bad_90d (更严重的逾期，是 30d 的子集)
y_90d = np.zeros_like(y_30d)
# 70% 的 30d 用户会转为 90d，但也加入少量随机性
mask_30 = (y_30d == 1)
y_90d[mask_30] = np.random.binomial(1, 0.7, mask_30.sum())

# 组装 DataFrame
df_sim = pd.DataFrame({
    "apply_date": apply_date,
    "bad_30d": y_30d,
    "bad_90d": y_90d,
    # 特征
    "age": age,
    "gender": gender,
    "education": education,
    "income_annual": income_annual,
    "house_value": house_value,
    "debt_ratio": debt_ratio,
    "max_overdue_days": max_overdue_days,
    "overdue_cnt_1y": overdue_cnt_1y,
    "credit_score_ext": credit_score_ext,
    "active_loans": active_loans,
    "query_cnt_3m": query_cnt_3m,
    "query_org_cnt": query_org_cnt,
    "util_rate": util_rate,
    "phone_in_net_months": phone_in_net_months,
    "height": height,
    "last_login_hour": last_login_hour,
    "device_type": device_type,
    "is_email_verified": is_email_verified,
    "app_install_cnt": app_install_cnt,
    "city_tier": city_tier
})

print(f"数据生成完毕: {df_sim.shape}")
print(f"Bad Rate (30d): {y_30d.mean():.2%}")
print(f"Bad Rate (90d): {y_90d.mean():.2%}")

# =================================================================
# 3. 运行 Profile Risk (多目标测试)
# =================================================================
target_cols = ["bad_30d", "bad_90d"]
# 自动识别除去 ID 和 Target 外的所有特征
feature_cols = [c for c in df_sim.columns if c not in target_cols and c != "apply_date"]

report = profile_risk(
    df=df_sim,
    target=target_cols,
    features=feature_cols,
    profile_by="month",
    dt_col="apply_date",
    
    # 策略设置
    binning_type="opt",  # 使用最优分箱
    n_bins=5,
    min_bin_size=0.05,
    monotonic_trend="auto_asc_desc", # 自动寻找单调趋势
    binner_kwargs= {
        "cat_features": ['device_type', 'gender', 'education']
    },
    
    plot=True # 多目标会自动关闭绘图，但会在日志提示
)

# =================================================================
# 4. 结果查看
# =================================================================
print("\n>>> Top 10 特征 (按 bad_30d IV 排序):")
# 展示 Summary，验证 IV 是否合理 (应在 0.02 - 0.5 之间)
summary = report.summary_table.to_pandas() if isinstance(report.summary_table, pl.DataFrame) else report.summary_table
summary_30d = summary[summary['target_col'] == 'bad_30d'].sort_values("IV_total", ascending=False).head(10)
display(summary_30d[["feature", "IV_total", "AUC_total", "KS_total", "Mars_Decision"]])

print("\n>>> 弱特征验证 (Height IV 应很低):")
print(summary[summary['feature'] == 'height'][["feature", "target_col", "IV_total", "Mars_Decision"]])

print("\n>>> 导出报告")
report.write_excel("mars_simulation_report.xlsx")

数据生成完毕: (200000, 23)
Bad Rate (30d): 5.00%
Bad Rate (90d): 3.50%
[MARS] 2026-02-09 01:56:41 - WARNING - ⚠️ Multi-target mode detected. Plotting is disabled to prevent UI freeze.
[MARS] 2026-02-09 01:56:41 - INFO - 🚀 Starting Risk Profiling [Primary Target: bad_30d | Strategy: OPT]
[MARS] 2026-02-09 01:56:41 - INFO - 👉 Evaluating Primary Target: bad_30d
[MARS] 2026-02-09 01:56:41 - INFO - ⚙️ No binner provided. Auto-fitting MarsOptimalBinner internally with params: {'n_bins': 5, 'min_bin_size': 0.05, 'monotonic_trend': 'auto_asc_desc', 'special_values': None, 'missing_values': None, 'n_jobs': -1, 'cat_features': ['device_type', 'gender', 'education']}...
[MARS] 2026-02-09 01:56:41 - INFO - ⏱️ [MarsNativeBinner.fit] finished in 0.4280s
[MARS] 2026-02-09 01:56:49 - INFO - ⏱️ [MarsOptimalBinner.fit] finished in 7.8357s
[MARS] 2026-02-09 01:56:49 - INFO - ⏱️ [MarsOptimalBinner.transform] finished in 0.0320s
[MARS] 2026-02-09 01:56:49 - INFO - ✅ Evaluation complete. [Features: 20 | Groups: 1

,feature,IV_total,AUC_total,KS_total,Mars_Decision
0,credit_score_ext,0.214951,0.624557,17.604737,✅ Keep: High Quality
1,phone_in_net_months,0.207408,0.625586,19.412632,✅ Keep: High Quality
2,max_overdue_days,0.204893,0.620513,18.121053,✅ Keep: High Quality
3,overdue_cnt_1y,0.081182,0.569503,11.894211,✅ Keep: High Quality
4,query_cnt_3m,0.045968,0.559773,9.047895,✅ Keep: High Quality
5,education,0.038897,0.550989,7.530526,✅ Keep: High Quality
6,income_annual,0.016043,0.533495,5.001053,🗑️ Drop: Weak Signal
7,debt_ratio,0.005159,0.518945,2.790526,❌ Drop: Logic Broken
8,age,0.002545,0.512678,1.976842,❌ Drop: Logic Broken
9,height,0.000514,0.505358,1.058947,❌ Drop: Logic Broken



>>> 弱特征验证 (Height IV 应很低):
   feature target_col  IV_total         Mars_Decision
9   height    bad_30d  0.000514  ❌ Drop: Logic Broken
34  height    bad_90d  0.000093  ❌ Drop: Logic Broken

>>> 导出报告
✅ [win32] 导出成功 (使用 xlwings)
